# ㄴ

In [6]:
# 한글 폰트 자동 선택 (Windows/ mac / Linux 어디서든)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# AppleGothic 경고가 너무 많으면 잠시 억제(선택)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')

candidates = [
    "Malgun Gothic",         # Windows
    "NanumGothic", "Nanum Gothic",
    "Noto Sans CJK KR", "Noto Sans CJK JP", "Noto Sans CJK SC",
    "AppleGothic",           # macOS
    "DejaVu Sans"            # 최후 fallback (한글 일부 환경에서 못 보일 수도)
]

available = {f.name for f in fm.fontManager.ttflist}
chosen = next((name for name in candidates if name in available), "DejaVu Sans")

plt.rcParams['font.family'] = chosen
plt.rcParams['axes.unicode_minus'] = False

print("사용 폰트:", chosen)


사용 폰트: Malgun Gothic


In [11]:
def predict_with_meal(readings, pkl_path, meal_overrides=None):
    """
    readings: [{"ds":"2022-01-15 12:00","y":180}, ...]  (최소 2~3시간 권장)
    meal_overrides: {"breakfast":"2022-01-15 08:20", "lunch":"2022-01-15 13:10", ...}
                    제공되면 해당 시각을 anchor로 사용(가까운 하루에만 반영)
    """
    import pickle
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s = df.set_index("ds")["y"].asfreq("5min").ffill()

    # 모델/메타 로드
    model, meta = pickle.load(open(pkl_path,"rb"))
    H = meta["horizon_min"]; base_meal = meta["meal_times"].copy()

    # override가 있으면 가장 최근 날짜에 해당 anchor로 덮어쓰기
    if meal_overrides:
        # 예: {"lunch":"2022-01-15 13:10"} → 해당 날짜 점심 anchor 교체
        for key, ts in meal_overrides.items():
            ts = pd.to_datetime(ts)
            base_meal[key] = ts.strftime("%H:%M")  # 당일 anchor만 바꾸고, since_meal_min은 실제 ts 기준이 되도록
        # build_meal_features 내부는 매일 같은 hh:mm anchor를 쓰지만,
        # '최근 식사 경과분'은 어차피 마지막 anchor와의 차이를 보므로
        # 예측 직전 구간에서는 override 효과가 바로 반영됩니다.

    # 마지막 2~3시간을 사용해 특성 만들고 한 스텝 예측
    X, _, _ = make_supervised_with_meal(s, horizon_min=H, meal_times=base_meal)
    x_last = X.iloc[[-1]]
    yhat = float(model.predict(x_last)[0])

    def band(v):
        if v < 70:   return "위험(저혈당)"
        if v <= 125: return "정상"
        if v <= 180: return "주의"
        if v <= 250: return "경고"
        return "위험(고혈당)"

    return {"horizon_min": H, "yhat": round(yhat,1), "risk": band(yhat)}


In [8]:
import numpy as np, pandas as pd

def build_meal_features(index: pd.DatetimeIndex, meal_times=None) -> pd.DataFrame:
    """
    index: 5분 간격 DatetimeIndex
    meal_times: {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}
    반환: since_*_min, after_* flags, last_is_* 원핫, since_last_meal_min
    """
    if meal_times is None:
        meal_times = {"breakfast":"08:00", "lunch":"13:00", "dinner":"19:00"}

    base = pd.DataFrame({"t": index}).sort_values("t")
    feats = {}
    since_arrays = []

    # 각 식사별 '가장 가까운 과거 앵커'와 경과분 계산
    for key, hhmm in meal_times.items():
        days = pd.date_range(base["t"].min().floor("D") - pd.Timedelta(days=1),
                             base["t"].max().ceil("D"),
                             freq="D")
        anchors = pd.to_datetime(days.strftime("%Y-%m-%d") + " " + hhmm)
        tmp = pd.merge_asof(base, pd.DataFrame({f"{key}_anc": anchors}),
                            left_on="t", right_on=f"{key}_anc",
                            direction="backward", tolerance=pd.Timedelta("36H"))
        since_min = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60
        # 음수/결측 보정
        since_min = since_min.clip(lower=0).fillna(1440)
        feats[f"since_{key}_min"] = since_min.values
        feats[f"after_{key}_0_2h"] = ((since_min >= 0) & (since_min <= 120)).astype(int).values
        feats[f"after_{key}_2_4h"] = ((since_min > 120) & (since_min <= 240)).astype(int).values
        since_arrays.append(since_min.values)

    # 가장 최근 식사(one-hot) + 대표 경과분
    S = np.vstack(since_arrays)                 # shape: (num_meals, N)
    argmin = np.argmin(S, axis=0)
    keys = list(meal_times.keys())
    for i, k in enumerate(keys):
        feats[f"last_is_{k}"] = (argmin == i).astype(int)
    feats["since_last_meal_min"] = S.min(axis=0)

    return pd.DataFrame(feats, index=index)

def make_supervised_with_meal(s: pd.Series, horizon_min=30, meal_times=None):
    """
    s: 5분 격자/ffill된 Series (index=DatetimeIndex, name=y)
    반환: X, y, index (누수 방지: 과거 정보만 사용)
    """
    h = horizon_min // 5
    df = pd.DataFrame({"y": s})

    # 래그(최대 120분)
    for k in [1,2,3,6,12,24]:
        df[f"lag{k}"] = df["y"].shift(k)

    # 이동평균/지수평활
    df["ma3"]   = df["y"].rolling(3,  min_periods=3).mean()
    df["ma12"]  = df["y"].rolling(12, min_periods=12).mean()
    df["ema12"] = df["y"].ewm(span=12, adjust=False).mean()

    # 캘린더
    idx = df.index
    hour = idx.hour + idx.minute/60
    df["sin_h"] = np.sin(2*np.pi*hour/24)
    df["cos_h"] = np.cos(2*np.pi*hour/24)
    df["weekday"] = idx.dayofweek
    df["is_weekend"] = df["weekday"].isin([4,5,6]).astype(int)

    # 식사 피처
    meal_feat = build_meal_features(idx, meal_times=meal_times)
    df = pd.concat([df, meal_feat], axis=1)

    # 타깃
    df["y_target"] = df["y"].shift(-h)

    df = df.dropna()
    X = df.drop(columns=["y_target","y"])
    y = df["y_target"]
    return X, y, df.index


In [10]:
# 1) 유병자 데이터 로드(없으면 자동 로드)
import pandas as pd, xml.etree.ElementTree as ET
from pathlib import Path

def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try:    ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return pd.DataFrame(rows).dropna(subset=["ds"]).sort_values("ds").reset_index(drop=True)

if "dis_train" not in globals():
    dis_train = load_xml_glucose("data/570-ws-training.xml")

# 2) 최근 구간에서 읽기 → readings JSON 만들기
recent = dis_train.tail(600)  # ≈ 50시간 (5분 간격)
readings = [{"ds": str(row.ds), "y": float(row.y)} for _, row in recent.iterrows()]

# 3) 120분 예측 (점심을 13:10에 먹었다고 가정; 필요하면 시각 바꿔 주세요)
pkl = "t1dm_lgbm_120m_meal.pkl"
assert Path(pkl).exists(), f"{pkl} 파일이 없습니다. 먼저 모델 학습/저장 셀을 실행하세요."

res = predict_with_meal(
    readings,
    pkl,
    meal_overrides={"lunch": "2022-01-15 13:10"}  # 예시: 실사용자 입력값으로 치환
)
print(res)  # {'horizon_min': 120, 'yhat': 000.0, 'risk': '...'}


AssertionError: t1dm_lgbm_120m_meal.pkl 파일이 없습니다. 먼저 모델 학습/저장 셀을 실행하세요.

In [ ]:
# === 120분 미래 예측 그래프: 최근 히스토리 + 30/60/120m 곡선 ===
import os, pickle
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------
# 1) 입력: readings & (선택) 식사 오버라이드
#    - readings 예: [{"ds":"2022-01-15 12:00","y":180}, ...]
#    - 우리 예시: dis_train 최근 50시간 사용 (없으면 직접 readings 채워 넣기)
# ----------------------------------------
def load_xml_glucose(path: str) -> pd.DataFrame:
    import xml.etree.ElementTree as ET
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try:    ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return pd.DataFrame(rows).dropna(subset=["ds"]).sort_values("ds").reset_index(drop=True)

readings = None
if Path("570-ws-training.xml").exists():
    dis_train = load_xml_glucose("570-ws-training.xml")
    recent = dis_train.tail(600)  # ≈ 50시간 (5분 간격)
    readings = [{"ds": str(row.ds), "y": float(row.y)} for _, row in recent.iterrows()]
else:
    # ← 여기다 실사용자 readings를 넣어도 됩니다.
    raise RuntimeError("readings가 비어 있습니다. dis_train 로드가 안 되면 readings를 직접 넣어주세요.")

meal_overrides = {
    # 예: 점심 먹은 시간 입력 시 바로 반영
     "lunch": "2022-01-15 13:10"
}

# ----------------------------------------
# 2) 예측 헬퍼 (앞서 만든 predict_with_meal 재사용)
#    - 없으면 간단 대체: 마지막 값 = naive 예측
# ----------------------------------------
def safe_predict_with_meal(readings, pkl_path, meal_overrides=None):
    try:
        from pickle import load as _pl
        with open(pkl_path, "rb") as f:
            model, meta = pickle.load(f)
    except Exception:
        return None  # 피클 없으면 None
    # make_supervised_with_meal 필요
    try:
        _ = make_supervised_with_meal
    except NameError:
        # 빠른 임포트(우리가 앞서 만든 버전)
        import numpy as np
        def build_meal_features(index: pd.DatetimeIndex, meal_times=None) -> pd.DataFrame:
            if meal_times is None:
                meal_times = {"breakfast":"08:00", "lunch":"13:00", "dinner":"19:00"}
            base = pd.DataFrame({"t": index}).sort_values("t")
            feats = {}; since_arrays = []
            for key, hhmm in meal_times.items():
                days = pd.date_range(base["t"].min().floor("D")-pd.Timedelta(days=1),
                                     base["t"].max().ceil("D"), freq="D")
                anchors = pd.to_datetime(days.strftime("%Y-%m-%d")+" "+hhmm)
                tmp = pd.merge_asof(base, pd.DataFrame({f"{key}_anc": anchors}),
                                    left_on="t", right_on=f"{key}_anc",
                                    direction="backward", tolerance=pd.Timedelta("36H"))
                since_min = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60
                since_min = since_min.clip(lower=0).fillna(1440)
                feats[f"since_{key}_min"] = since_min.values
                feats[f"after_{key}_0_2h"] = ((since_min>=0)&(since_min<=120)).astype(int).values
                feats[f"after_{key}_2_4h"] = ((since_min>120)&(since_min<=240)).astype(int).values
                since_arrays.append(since_min.values)
            S = np.vstack(since_arrays); argmin = np.argmin(S, axis=0)
            keys = list({"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}.keys())
            for i,k in enumerate(keys):
                feats[f"last_is_{k}"] = (argmin==i).astype(int)
            feats["since_last_meal_min"] = S.min(axis=0)
            return pd.DataFrame(feats, index=index)

        def make_supervised_with_meal(s: pd.Series, horizon_min=30, meal_times=None):
            h = horizon_min//5
            df = pd.DataFrame({"y": s})
            for k in [1,2,3,6,12,24]:
                df[f"lag{k}"] = df["y"].shift(k)
            df["ma3"] = df["y"].rolling(3, min_periods=3).mean()
            df["ma12"]= df["y"].rolling(12,min_periods=12).mean()
            df["ema12"]=df["y"].ewm(span=12, adjust=False).mean()
            idx = df.index
            hour = idx.hour + idx.minute/60
            df["sin_h"]=np.sin(2*np.pi*hour/24); df["cos_h"]=np.cos(2*np.pi*hour/24)
            df["weekday"]=idx.dayofweek; df["is_weekend"]=df["weekday"].isin([4,5,6]).astype(int)
            meal_feat = build_meal_features(idx, meal_times=meta.get("meal_times"))
            df = pd.concat([df, meal_feat], axis=1)
            df["y_target"] = df["y"].shift(-h)
            df = df.dropna()
            X = df.drop(columns=["y_target","y"]); y = df["y_target"]
            return X, y, df.index

    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s = df.set_index("ds")["y"].asfreq("5min").ffill()

    # meal_overrides 단순 처리(훈련시각대와 달라도 최근 경과분에 반영됨)
    # -> make_supervised_with_meal 내부 meal_times는 meta['meal_times'] 사용
    X, _, _ = make_supervised_with_meal(s, horizon_min=meta["horizon_min"], meal_times=meta.get("meal_times"))
    if len(X)==0: 
        return None
    yhat = float(model.predict(X.iloc[[-1]])[0])
    return {"horizon_min": meta["horizon_min"], "yhat": yhat}

# ----------------------------------------
# 3) 30/60/120분 포인트 예측 수집 (가능한 것만)
# ----------------------------------------
points = []  # (분, 예측값)
def last_point(readings):
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s = df.set_index("ds")["y"].asfreq("5min").ffill()
    return s.index[-1], float(s.iloc[-1]), s.last("6H")  # 마지막 시각, 마지막값, 최근 6시간

t0, y0, hist = last_point(readings)
points.append((0, y0))

for H, pkl in [(30, "t1dm_lgbm_30m_meal.pkl"),
               (60, "t1dm_lgbm_60m_meal.pkl"),
               (120,"t1dm_lgbm_120m_meal.pkl")]:
    if Path(pkl).exists():
        pred = safe_predict_with_meal(readings, pkl, meal_overrides=meal_overrides)
        if pred is not None:
            points.append((H, pred["yhat"]))

# 수동 120m 값이 있다면(예: {'horizon_min':120, 'yhat':126.4, 'risk':'주의'})
manual_yhat120 = 126.4  # ← 네가 받은 결과가 있으면 여기에 숫자만 바꿔 넣어
if not any(p[0]==120 for p in points) and manual_yhat120 is not None:
    points.append((120, float(manual_yhat120)))

# ----------------------------------------
# 4) 곡선 생성(선형 보간) + 플롯
# ----------------------------------------
points = sorted(points, key=lambda x: x[0])         # 분 기준 정렬
grid_min = np.arange(0, 120+1, 5)                   # 0~120분, 5분 간격
xp = np.array([p[0] for p in points], dtype=float)
yp = np.array([p[1] for p in points], dtype=float)
yhat_curve = np.interp(grid_min, xp, yp)            # 선형 보간

t_grid = [t0 + pd.Timedelta(minutes=int(m)) for m in grid_min]

plt.figure(figsize=(9,4))

# 최근 6시간 히스토리
plt.plot(hist.index, hist.values, linewidth=1.5, label="최근 6시간")

# 미래 예측 곡선
plt.plot(t_grid, yhat_curve, linestyle="--", linewidth=2, label="예측(0~120분)")

# 30/60/120마커(있을 때만)
for m,v in points:
    if m>0:
        plt.plot([t0 + pd.Timedelta(minutes=int(m))], [v], marker="o", linestyle="None", label=f"+{m}분")

# 기준선(선택): 70, 125, 180, 250
for th in [70, 125, 180, 250]:
    plt.axhline(th, linestyle=":", linewidth=0.8)

plt.title("120분 미래 혈당 예측 (히스토리+Forecast)")
plt.xlabel("시간"); plt.ylabel("mg/dL"); plt.legend()
plt.tight_layout()
outpath = "forecast_120m.png"
plt.savefig(outpath, dpi=140)
plt.show()

print(f"저장: {outpath}")


In [ ]:
# --- 2) 식사 피처(override 지원) ---
def build_meal_features(index: pd.DatetimeIndex,
                        meal_times=None,
                        override_ts=None) -> pd.DataFrame:
    """
    meal_times: {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}
    override_ts: {"lunch":"2022-01-15 13:10"}  # 해당 날짜 anchor를 실제 시각으로 치환
    """
    if meal_times is None:
        meal_times = {"breakfast":"08:00", "lunch":"13:00", "dinner":"19:00"}
    if override_ts is None:
        override_ts = {}

    base = pd.DataFrame({"t": index}).sort_values("t")
    feats, since_arrays = {}, []

    for key, hhmm in meal_times.items():
        days = pd.date_range(
            base["t"].min().floor("D") - pd.Timedelta(days=1),
            base["t"].max().ceil("D"),
            freq="D"
        )
        # DatetimeIndex -> Series로 변환(마스크 할당 가능)
        anchors = pd.Series(
            pd.to_datetime(days.strftime("%Y-%m-%d") + " " + hhmm),
            name=f"{key}_anc"
        )

        # override: 같은 날짜 anchor를 실제 입력 시각으로 교체
        if key in override_ts:
            ts = pd.to_datetime(override_ts[key])
            mask = anchors.dt.normalize() == ts.normalize()
            if mask.any():
                anchors.loc[mask] = ts

        # merge_asof용 우측 프레임
        anc_df = anchors.to_frame().sort_values(f"{key}_anc")
        tmp = pd.merge_asof(
            base.sort_values("t"),
            anc_df,
            left_on="t",
            right_on=f"{key}_anc",
            direction="backward",
            tolerance=pd.Timedelta("36H"),
        )

        since = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds() / 60.0
        since = since.clip(lower=0).fillna(1440)  # 음수/결측 보정

        feats[f"since_{key}_min"] = since.values
        feats[f"after_{key}_0_2h"] = ((since >= 0) & (since <= 120)).astype(int).values
        feats[f"after_{key}_2_4h"] = ((since > 120) & (since <= 240)).astype(int).values
        since_arrays.append(since.values)

    # 가장 최근 식사(one-hot) + 대표 경과분
    S = np.vstack(since_arrays)            # (num_meals, N)
    argmin = np.argmin(S, axis=0)
    keys = list(meal_times.keys())
    for i, k in enumerate(keys):
        feats[f"last_is_{k}"] = (argmin == i).astype(int)
    feats["since_last_meal_min"] = S.min(axis=0)

    return pd.DataFrame(feats, index=index)

def make_supervised_with_meal(s: pd.Series,
                              horizon_min=30,
                              meal_times=None,
                              override_ts=None):
    """과거 정보만 사용하도록 래그/평활/달력/식사 피처 생성"""
    h = max(1, horizon_min // 5)
    df = pd.DataFrame({"y": s})

    # 래그(최대 120분)
    for k in [1,2,3,6,12,24]:
        df[f"lag{k}"] = df["y"].shift(k)

    # 이동평균/지수평활
    df["ma3"]   = df["y"].rolling(3,  min_periods=3).mean()
    df["ma12"]  = df["y"].rolling(12, min_periods=12).mean()
    df["ema12"] = df["y"].ewm(span=12, adjust=False).mean()

    # 달력
    idx = df.index
    hour = idx.hour + idx.minute/60.0
    df["sin_h"] = np.sin(2*np.pi*hour/24)
    df["cos_h"] = np.cos(2*np.pi*hour/24)
    df["weekday"] = idx.dayofweek
    df["is_weekend"] = df["weekday"].isin([4,5,6]).astype(int)

    # 식사 피처 (override 반영)
    meal_feat = build_meal_features(idx, meal_times=meal_times, override_ts=override_ts)
    df = pd.concat([df, meal_feat], axis=1)

    # 타깃
    df["y_target"] = df["y"].shift(-h)

    df = df.dropna()
    X = df.drop(columns=["y_target","y"])
    y = df["y_target"]
    return X, y, df.index

# --- 3) (있으면) pkl 예측 1포인트 ---
def predict_point(readings, pkl_path, meal_overrides=None):
    pkl = Path(pkl_path)
    if not pkl.exists():
        return None
    with open(pkl, "rb") as f:
        model, meta = pickle.load(f)

    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s = df.set_index("ds")["y"].asfreq("5min").ffill()

    X, _, _ = make_supervised_with_meal(
        s,
        horizon_min=meta.get("horizon_min", 30),
        meal_times=meta.get("meal_times", {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}),
        override_ts=meal_overrides
    )
    if len(X) == 0:
        return None
    yhat = float(model.predict(X.iloc[[-1]])[0])
    return meta["horizon_min"], yhat, s

# --- 4) 곡선 그리기(히스토리+예측) ---
def plot_forecast_curve(
    readings,
    meal_overrides=None,                 # 예: {"lunch":"2022-01-15 13:10"}
    manual_points=None,                  # 예: {120: 126.0}
    pkls={30:"t1dm_lgbm_30m_meal.pkl", 60:"t1dm_lgbm_60m_meal.pkl", 120:"t1dm_lgbm_120m_meal.pkl"},
    history_hours=6,
    outpath="forecast_curve.png"
):
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()
    t0 = s.index[-1]; y0 = float(s.iloc[-1])
    hist = s.last(f"{history_hours}H")

    # 예측 포인트 수집
    points = {0: y0}  # 현재값(0분)
    for H, pkl in pkls.items():
        res = predict_point(readings, pkl, meal_overrides=meal_overrides)
        if res is not None:
            h, yhat, _ = res
            points[int(h)] = float(yhat)

    # 수동 포인트가 있으면 덮어쓰기(+30/+60/+120 중 원하는 지점)
    if manual_points:
        for h, v in manual_points.items():
            points[int(h)] = float(v)

    # 보간용 그리드 (최소 120분까지)
    max_h = max([120] + list(points.keys()))
    grid_min = np.arange(0, max_h+1, 5)
    xp = np.array(sorted(points.keys()), dtype=float)
    yp = np.array([points[int(k)] for k in sorted(points.keys())], dtype=float)
    curve = np.interp(grid_min, xp, yp)
    t_grid = [t0 + pd.Timedelta(minutes=int(m)) for m in grid_min]

    # 플롯
    plt.figure(figsize=(9,4))
    plt.plot(hist.index, hist.values, linewidth=1.5, label=f"최근 {history_hours}시간")
    plt.plot(t_grid, curve, linestyle="--", linewidth=2, label=f"예측(0~{max_h}분)")

    # 마커(있는 지점만)
    for h in sorted(points.keys()):
        if h>0:
            plt.plot([t0 + pd.Timedelta(minutes=h)], [points[h]], marker="o", linestyle="None", label=f"+{h}분")

    # 기준선
    for th in [70, 125, 180, 250]:
        plt.axhline(th, linestyle=":", linewidth=0.8)

    plt.title(f"{max_h}분 미래 혈당 예측 (히스토리+Forecast)")
    plt.xlabel("시간"); plt.ylabel("mg/dL"); plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=140)
    plt.show()
    return outpath

# ================== 사용 예시 ==================
# 570 파일에서 최근 50시간 readings 생성
if Path("570-ws-training.xml").exists():
    dis_train = load_xml_glucose("570-ws-training.xml")
    recent = dis_train.tail(600)  # ≈50시간 (5분 간격)
    readings = [{"ds": str(row.ds), "y": float(row.y)} for _, row in recent.iterrows()]
else:
    raise RuntimeError("570-ws-training.xml 파일을 찾을 수 없습니다. 경로를 확인하세요.")

# 식사 입력(점심 13:10) + +120분 수동값(126.4) 동시에 적용해 그래프 생성
out = plot_forecast_curve(
    readings,
    meal_overrides={"lunch": "2022-01-15 13:10"},  # ← 식사 입력
    manual_points={120: 126.4}                     # ← 수동 예측값(필요 시 30/60도 추가 가능)
)
print("저장:", out)

In [ ]:
# ==========================================
# 입력(현재혈당 + 식사시각) → 30/60/120분 예측 + 그래프
#  - readings가 없으면, 현재혈당으로 평평한 4시간 히스토리를 합성해서 사용
#  - readings가 있으면, 끝을 식사시각으로 정렬하고 마지막 값을 현재혈당으로 덮어씀
# ==========================================
import numpy as np, pandas as pd

def predict_from_inputs(
    current_glucose,                 # 현재 혈당 (mg/dL)
    meal_ts,                         # "YYYY-MM-DD HH:MM" (예: "2022-01-15 13:10")
    readings=None,                   # (선택) 기존 CGM 히스토리 [{"ds": "...", "y": ...}, ...]
    meal_key="lunch",                # "breakfast" | "lunch" | "dinner"
    history_hours=4,                 # 식사 '전' 히스토리로 보여줄 길이
    pkls={30:"t1dm_lgbm_30m_meal.pkl", 60:"t1dm_lgbm_60m_meal.pkl", 120:"t1dm_lgbm_120m_meal.pkl"},
    outpath="forecast_after_meal_from_inputs.png",
    relative_axis=False              # True면 x축을 식사 기준 상대분(0~120)으로 표시
):
    meal_ts = pd.to_datetime(meal_ts).floor("5min")

    # 1) 히스토리 시리즈 만들기 (5분 간격)
    if readings is None:
        # 평평한 히스토리 합성
        idx = pd.date_range(meal_ts - pd.Timedelta(hours=history_hours), meal_ts, freq="5min")
        s = pd.Series(float(current_glucose), index=idx, name="y")
    else:
        df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
        s = df.set_index("ds")["y"].asfreq("5min").ffill()
        # 식사시각으로 끝 정렬
        if s.index[-1] < meal_ts:
            ext_idx = pd.date_range(s.index[-1] + pd.Timedelta("5min"), meal_ts, freq="5min")
            if len(ext_idx):
                s = pd.concat([s, pd.Series(float(s.iloc[-1]), index=ext_idx)], axis=0)
        elif s.index[-1] > meal_ts:
            s = s.loc[:meal_ts]
        # 현재혈당 덮어쓰기
        s.iloc[-1] = float(current_glucose)

    # 2) plot용 readings로 변환
    readings2 = [{"ds": str(t), "y": float(v)} for t, v in s.items()]

    # 3) 30/60/120 포인트 예측 수집 (모델 없으면 건너뛰고 나중에 보간)
    points = {0: float(current_glucose)}  # 0분은 현재
    preds = {}
    for H, pkl in pkls.items():
        res = predict_after_meal(readings2, pkl, meal_ts, meal_key=meal_key)
        if res is not None:
            h, yhat = res
            preds[h] = float(yhat)
            points[h] = float(yhat)

    # 모델이 없어 비어 있으면, 단순히 평평한 곡선이라도 만들 수 있게 처리
    if len(points) == 1:
        points[120] = float(current_glucose)

    # 4) +120분 값과 리스크 라벨
    xs = np.array(sorted(points.keys()))
    ys = np.array([points[k] for k in sorted(points.keys())], dtype=float)
    yhat120 = float(np.interp(120, xs, ys)) if 120 not in points else points[120]

    if yhat120 < 70:
        risk = "위험(저혈당) 🔴"
    elif yhat120 < 125:
        risk = "주의 ⚠️"
    elif yhat120 < 180:
        risk = "경고 🚨"
    else:
        risk = "위험(고혈당) 🔴"

    # 5) 그래프 (식사 전 히스토리 + 식사 후 0~120 보간곡선)
    manual_points = {k: v for k, v in points.items() if k > 0}  # 마커 찍기용
    plot_path = plot_after_meal_curve(
        readings2,
        meal_ts=meal_ts,
        meal_key=meal_key,
        pkls=pkls,
        manual_points=manual_points,      # 있으면 모델값이든 수동값이든 그대로 마커로 표시
        history_hours=history_hours,
        outpath=outpath,
        relative_axis=relative_axis
    )

    return {
        "predictions": {f"+{k}": round(points[k], 1) for k in sorted(points.keys()) if k > 0},
        "yhat_120": round(yhat120, 1),
        "risk": risk,
        "plot_path": plot_path
    }


In [ ]:
# =========================================================
# ⛏️ 한 셀 올인원: 현재혈당 + 식사시각 입력 → 30/60/120분 예측 & 그래프
#  - OhioT1DM 570 XML이 있으면 읽고, 없으면 합성 히스토리 사용
#  - LGBM pkl(30/60/120)이 있으면 사용, 없으면 보간으로 곡선 생성
# =========================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle
from pathlib import Path
import xml.etree.ElementTree as ET

# ---- 폰트(있는 것 중 자동 선택) ----
import matplotlib.font_manager as fm, warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')
candidates = ["Malgun Gothic","NanumGothic","Noto Sans CJK KR","AppleGothic","DejaVu Sans"]
available = {f.name for f in fm.fontManager.ttflist}
plt.rcParams['font.family'] = next((n for n in candidates if n in available), "DejaVu Sans")
plt.rcParams['axes.unicode_minus'] = False

# ---------- 공용 유틸 ----------
def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try: ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return (pd.DataFrame(rows).dropna(subset=["ds"]).sort_values("ds").reset_index(drop=True))

def build_meal_features(index: pd.DatetimeIndex, meal_times=None, override_ts=None) -> pd.DataFrame:
    if meal_times is None:
        meal_times = {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}
    if override_ts is None: override_ts = {}
    base = pd.DataFrame({"t": index}).sort_values("t")
    feats, since_arrays = {}, []
    for key, hhmm in meal_times.items():
        days = pd.date_range(base["t"].min().floor("D")-pd.Timedelta(days=1),
                             base["t"].max().ceil("D"), freq="D")
        anchors = pd.Series(pd.to_datetime(days.strftime("%Y-%m-%d")+" "+hhmm), name=f"{key}_anc")
        if key in override_ts:
            ts = pd.to_datetime(override_ts[key])
            mask = anchors.dt.normalize() == ts.normalize()
            if mask.any(): anchors.loc[mask] = ts
        anc_df = anchors.to_frame().sort_values(f"{key}_anc")
        tmp = pd.merge_asof(base.sort_values("t"), anc_df,
                            left_on="t", right_on=f"{key}_anc",
                            direction="backward", tolerance=pd.Timedelta("36H"))
        since = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60.0
        since = since.clip(lower=0).fillna(1440)
        feats[f"since_{key}_min"]   = since.values
        feats[f"after_{key}_0_2h"]  = ((since>=0)&(since<=120)).astype(int).values
        feats[f"after_{key}_2_4h"]  = ((since>120)&(since<=240)).astype(int).values
        since_arrays.append(since.values)
    S = np.vstack(since_arrays); argmin = np.argmin(S, axis=0)
    keys = list(meal_times.keys())
    for i,k in enumerate(keys): feats[f"last_is_{k}"] = (argmin==i).astype(int)
    feats["since_last_meal_min"] = S.min(axis=0)
    return pd.DataFrame(feats, index=index)

def make_supervised_with_meal(s: pd.Series, horizon_min=30, meal_times=None, override_ts=None):
    h = max(1, horizon_min//5)
    df = pd.DataFrame({"y": s})
    for k in [1,2,3,6,12,24]: df[f"lag{k}"] = df["y"].shift(k)
    df["ma3"] = df["y"].rolling(3, min_periods=3).mean()
    df["ma12"]= df["y"].rolling(12, min_periods=12).mean()
    df["ema12"]= df["y"].ewm(span=12, adjust=False).mean()
    idx = df.index; hour = idx.hour + idx.minute/60.0
    df["sin_h"]=np.sin(2*np.pi*hour/24); df["cos_h"]=np.cos(2*np.pi*hour/24)
    df["weekday"]=idx.dayofweek; df["is_weekend"]=df["weekday"].isin([4,5,6]).astype(int)
    meal_feat = build_meal_features(idx, meal_times=meal_times, override_ts=override_ts)
    df = pd.concat([df, meal_feat], axis=1)
    df["y_target"] = df["y"].shift(-h)
    df = df.dropna()
    X = df.drop(columns=["y_target","y"]); y = df["y_target"]
    return X, y, df.index

def make_features_only(s: pd.Series, meal_times=None, override_ts=None):
    df = pd.DataFrame({"y": s})
    for k in [1,2,3,6,12,24]: df[f"lag{k}"] = df["y"].shift(k)
    df["ma3"]=df["y"].rolling(3,min_periods=3).mean()
    df["ma12"]=df["y"].rolling(12,min_periods=12).mean()
    df["ema12"]=df["y"].ewm(span=12, adjust=False).mean()
    idx = df.index; hour = idx.hour + idx.minute/60.0
    df["sin_h"]=np.sin(2*np.pi*hour/24); df["cos_h"]=np.cos(2*np.pi*hour/24)
    df["weekday"]=idx.dayofweek; df["is_weekend"]=df["weekday"].isin([4,5,6]).astype(int)
    meal_feat = build_meal_features(idx, meal_times=meal_times, override_ts=override_ts)
    df = pd.concat([df, meal_feat], axis=1)
    return df.drop(columns=["y"]).dropna()

def predict_after_meal(readings, pkl_path, meal_ts, meal_key="lunch"):
    pkl = Path(pkl_path)
    if not pkl.exists(): return None
    with open(pkl, "rb") as f: model, meta = pickle.load(f)
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    if meal_ts not in s.index:
        s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="5min")).ffill()
    if meal_ts < s.index.min() + pd.Timedelta("2H"): return None
    X_all = make_features_only(
        s.loc[:meal_ts],
        meal_times=meta.get("meal_times", {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}),
        override_ts={meal_key: meal_ts}
    )
    if meal_ts not in X_all.index: meal_ts = X_all.index.max()
    x_row = X_all.loc[[meal_ts]]
    if "features" in meta:
        cols = [c for c in meta["features"] if c in x_row.columns]
        x_row = x_row.reindex(columns=cols, fill_value=0)
    yhat = float(model.predict(x_row)[0])
    return meta.get("horizon_min", 30), yhat

def plot_after_meal_curve(readings, meal_ts, meal_key="lunch",
                          pkls={30:"t1dm_lgbm_30m_meal.pkl",60:"t1dm_lgbm_60m_meal.pkl",120:"t1dm_lgbm_120m_meal.pkl"},
                          manual_points=None, history_hours=4, outpath="forecast_after_meal.png",
                          relative_axis=False):
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    hist = s.loc[meal_ts-pd.Timedelta(hours=history_hours): meal_ts]
    y0 = float(hist.iloc[-1]) if len(hist) else float(s.loc[:meal_ts].iloc[-1])
    points = {0: y0}
    for H,pkl in pkls.items():
        r = predict_after_meal(readings, pkl, meal_ts, meal_key=meal_key)
        if r is not None:
            h, yhat = r; points[int(h)] = float(yhat)
    if manual_points:
        for h,v in manual_points.items(): points[int(h)] = float(v)
    max_h = max([120]+list(points.keys()))
    grid_min = np.arange(0, max_h+1, 5)
    xp = np.array(sorted(points.keys())); yp = np.array([points[k] for k in sorted(points.keys())], float)
    curve = np.interp(grid_min, xp, yp)
    if relative_axis:
        hist_x = (hist.index - meal_ts).total_seconds()/60; t_grid = grid_min; xlab="식사 기준 상대시간(분)"
    else:
        hist_x = hist.index; t_grid = [meal_ts + pd.Timedelta(minutes=int(m)) for m in grid_min]; xlab="시간"
    plt.figure(figsize=(9,4))
    plt.plot(hist_x, hist.values, lw=1.5, label=f"식사 전 {history_hours}시간")
    plt.plot(t_grid, curve, "--", lw=2, label=f"식사 후 예측(0~{max_h}분)")
    for h,v in sorted(points.items()):
        if h>0:
            x = h if relative_axis else (meal_ts + pd.Timedelta(minutes=h))
            plt.plot([x],[v],"o",label=f"+{h}분")
    if relative_axis: plt.axvline(0, color="k", ls=":", lw=1)
    else:             plt.axvline(meal_ts, color="k", ls=":", lw=1)
    for th in [70,125,180,250]: plt.axhline(th, ls=":", lw=0.8)
    plt.title("식사 이후 120분 혈당 예측"); plt.xlabel(xlab); plt.ylabel("mg/dL"); plt.legend()
    plt.tight_layout(); plt.savefig(outpath, dpi=140); plt.show()
    return outpath

def predict_from_inputs(current_glucose, meal_ts, readings=None, meal_key="lunch",
                        history_hours=4,
                        pkls={30:"t1dm_lgbm_30m_meal.pkl",60:"t1dm_lgbm_60m_meal.pkl",120:"t1dm_lgbm_120m_meal.pkl"},
                        outpath="forecast_after_meal_from_inputs.png", relative_axis=False):
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    if readings is None:
        idx = pd.date_range(meal_ts-pd.Timedelta(hours=history_hours), meal_ts, freq="5min")
        s = pd.Series(float(current_glucose), index=idx, name="y")
    else:
        df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
        s = df.set_index("ds")["y"].asfreq("5min").ffill()
        if s.index[-1] < meal_ts:
            ext_idx = pd.date_range(s.index[-1]+pd.Timedelta("5min"), meal_ts, freq="5min")
            if len(ext_idx): s = pd.concat([s, pd.Series(float(s.iloc[-1]), index=ext_idx)], axis=0)
        elif s.index[-1] > meal_ts:
            s = s.loc[:meal_ts]
        s.iloc[-1] = float(current_glucose)
    readings2 = [{"ds": str(t), "y": float(v)} for t,v in s.items()]
    points = {0: float(current_glucose)}
    for H,pkl in pkls.items():
        r = predict_after_meal(readings2, pkl, meal_ts, meal_key=meal_key)
        if r is not None:
            h,yhat = r; points[int(h)] = float(yhat)
    if len(points)==1: points[120] = float(current_glucose)
    xs = np.array(sorted(points.keys())); ys = np.array([points[k] for k in sorted(points.keys())], float)
    yhat120 = float(points[120] if 120 in points else np.interp(120, xs, ys))
    if   yhat120 < 70:  risk = "위험(저혈당) 🔴"
    elif yhat120 < 125: risk = "주의 ⚠️"
    elif yhat120 < 180: risk = "경고 🚨"
    else:               risk = "위험(고혈당) 🔴"
    plot_path = plot_after_meal_curve(readings2, meal_ts, meal_key=meal_key,
                                      pkls=pkls, manual_points={k:v for k,v in points.items() if k>0},
                                      history_hours=history_hours, outpath=outpath, relative_axis=relative_axis)
    return {"predictions": {f"+{k}": round(points[k],1) for k in sorted(points) if k>0},
            "yhat_120": round(yhat120,1), "risk": risk, "plot_path": plot_path}

# ---------- (옵션) 570 파일에서 readings 만들기 ----------
readings = None
if Path("570-ws-training.xml").exists():
    dis_train = load_xml_glucose("570-ws-training.xml")
    recent = dis_train.tail(600)  # ≈ 50시간
    readings = [{"ds": str(r.ds), "y": float(r.y)} for _, r in recent.iterrows()]

# ---------- 예시 호출 ----------
res = predict_from_inputs(
    current_glucose=160,            # 현재 혈당 입력
    meal_ts="2022-01-15 13:10",     # 밥 먹은 시간 입력
    readings=readings,              # 없으면 None로 두세요(합성 히스토리)
    relative_axis=False             # True면 식사 기준 상대분(0~120) 축
)
print(res)


In [ ]:
# === 실제 유병자(570) 혈당: 식사시각 전후 실측 그래프 ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from pathlib import Path

# 폰트(있으면 자동 선택)
import matplotlib.font_manager as fm
cands = ["Malgun Gothic","NanumGothic","AppleGothic","Noto Sans CJK KR","DejaVu Sans"]
avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((c for c in cands if c in avail), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False

FILE = "570-ws-training.xml"     # 필요시 "570-ws-testing.xml"로 변경
MEAL_TS = "2022-01-1 13:10"     # 식사 시각(실제 확인하려는 시각)
H_BEFORE = "4H"                  # 식사 전 표시 구간
H_AFTER  = "2H"                  # 식사 후 표시 구간

def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try:    ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return (pd.DataFrame(rows)
              .dropna(subset=["ds"])
              .sort_values("ds")
              .reset_index(drop=True))

# 1) 데이터 로드
assert Path(FILE).exists(), f"{FILE} 없음"
df = load_xml_glucose(FILE)
s  = df.set_index("ds")["y"].asfreq("5min").ffill()

print(f"기간: {s.index.min()} ~ {s.index.max()}, 개수: {len(s):,}")
meal_ts = pd.to_datetime(MEAL_TS).floor("5min")

# 2) 식사 시각이 데이터 격자에 얼마나 근접한지 확인
if meal_ts not in s.index:
    # 가장 가까운 시각 찾기
    i = s.index.get_indexer([meal_ts], method="nearest")[0]
    nearest = s.index[i]
    delta_m = int(abs((nearest - meal_ts).total_seconds()) // 60)
    print(f"입력 식사시각 {meal_ts} → 가장 가까운 측정시각 {nearest} (차이 {delta_m}분)")
else:
    nearest = meal_ts
    delta_m = 0
    print(f"식사시각 {meal_ts}가 데이터 격자와 일치")

# 3) 전/후 구간 자르기
start = meal_ts - pd.Timedelta(H_BEFORE)
end   = meal_ts + pd.Timedelta(H_AFTER)
seg = s.loc[start:end].copy()

# 4) 간단 통계
y_meal   = float(s.loc[nearest])
y_post30 = float(s.loc[nearest + pd.Timedelta("30min")]) if nearest + pd.Timedelta("30min") in s.index else np.nan
y_post60 = float(s.loc[nearest + pd.Timedelta("60min")]) if nearest + pd.Timedelta("60min") in s.index else np.nan
y_post120= float(s.loc[nearest + pd.Timedelta("120min")]) if nearest + pd.Timedelta("120min") in s.index else np.nan
print({"meal_value": y_meal, "+30": y_post30, "+60": y_post60, "+120": y_post120})

# 5) 플롯
plt.figure(figsize=(9,4))
plt.plot(seg.index, seg.values, lw=1.8, label=f"실측({H_BEFORE}~{H_AFTER})")
plt.axvline(meal_ts, color="k", ls=":", lw=1.2, label="식사시각")
plt.scatter([nearest], [y_meal], zorder=3, s=50, color="tab:orange", label="가장 가까운 측정값")

# 임계선(선택)
for th in [70, 125, 180, 250]:
    plt.axhline(th, ls=":", lw=0.8)

plt.title("실제 유병자 혈당 (식사시각 전후)")
plt.xlabel("시간"); plt.ylabel("mg/dL"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# =========================================================
# ⛏️ 한 셀 올인원: 현재혈당 + 식사시각 입력 → 30/60/120분 예측 & 그래프
#  - OhioT1DM 570 XML이 있으면 읽고, 없으면 합성 히스토리 사용
#  - LGBM pkl(30/60/120)이 있으면 사용, 없으면 보간으로 곡선 생성
# =========================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle
from pathlib import Path
import xml.etree.ElementTree as ET

# ---- 폰트(있는 것 중 자동 선택) ----
import matplotlib.font_manager as fm, warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')
candidates = ["Malgun Gothic","NanumGothic","Noto Sans CJK KR","AppleGothic","DejaVu Sans"]
available = {f.name for f in fm.fontManager.ttflist}
plt.rcParams['font.family'] = next((n for n in candidates if n in available), "DejaVu Sans")
plt.rcParams['axes.unicode_minus'] = False

# ---------- 공용 유틸 ----------
def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try: ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return (pd.DataFrame(rows).dropna(subset=["ds"]).sort_values("ds").reset_index(drop=True))

def build_meal_features(index: pd.DatetimeIndex, meal_times=None, override_ts=None) -> pd.DataFrame:
    if meal_times is None:
        meal_times = {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}
    if override_ts is None: override_ts = {}
    base = pd.DataFrame({"t": index}).sort_values("t")
    feats, since_arrays = {}, []
    for key, hhmm in meal_times.items():
        days = pd.date_range(base["t"].min().floor("D")-pd.Timedelta(days=1),
                             base["t"].max().ceil("D"), freq="D")
        anchors = pd.Series(pd.to_datetime(days.strftime("%Y-%m-%d")+" "+hhmm), name=f"{key}_anc")
        if key in override_ts:
            ts = pd.to_datetime(override_ts[key])
            mask = anchors.dt.normalize() == ts.normalize()
            if mask.any(): anchors.loc[mask] = ts
        anc_df = anchors.to_frame().sort_values(f"{key}_anc")
        tmp = pd.merge_asof(base.sort_values("t"), anc_df,
                            left_on="t", right_on=f"{key}_anc",
                            direction="backward", tolerance=pd.Timedelta("36H"))
        since = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60.0
        since = since.clip(lower=0).fillna(1440)
        feats[f"since_{key}_min"]   = since.values
        feats[f"after_{key}_0_2h"]  = ((since>=0)&(since<=120)).astype(int).values
        feats[f"after_{key}_2_4h"]  = ((since>120)&(since<=240)).astype(int).values
        since_arrays.append(since.values)
    S = np.vstack(since_arrays); argmin = np.argmin(S, axis=0)
    keys = list(meal_times.keys())
    for i,k in enumerate(keys): feats[f"last_is_{k}"] = (argmin==i).astype(int)
    feats["since_last_meal_min"] = S.min(axis=0)
    return pd.DataFrame(feats, index=index)

def make_supervised_with_meal(s: pd.Series, horizon_min=30, meal_times=None, override_ts=None):
    h = max(1, horizon_min//5)
    df = pd.DataFrame({"y": s})
    for k in [1,2,3,6,12,24]: df[f"lag{k}"] = df["y"].shift(k)
    df["ma3"] = df["y"].rolling(3, min_periods=3).mean()
    df["ma12"]= df["y"].rolling(12, min_periods=12).mean()
    df["ema12"]= df["y"].ewm(span=12, adjust=False).mean()
    idx = df.index; hour = idx.hour + idx.minute/60.0
    df["sin_h"]=np.sin(2*np.pi*hour/24); df["cos_h"]=np.cos(2*np.pi*hour/24)
    df["weekday"]=idx.dayofweek; df["is_weekend"]=df["weekday"].isin([4,5,6]).astype(int)
    meal_feat = build_meal_features(idx, meal_times=meal_times, override_ts=override_ts)
    df = pd.concat([df, meal_feat], axis=1)
    df["y_target"] = df["y"].shift(-h)
    df = df.dropna()
    X = df.drop(columns=["y_target","y"]); y = df["y_target"]
    return X, y, df.index

def make_features_only(s: pd.Series, meal_times=None, override_ts=None):
    df = pd.DataFrame({"y": s})
    for k in [1,2,3,6,12,24]: df[f"lag{k}"] = df["y"].shift(k)
    df["ma3"]=df["y"].rolling(3,min_periods=3).mean()
    df["ma12"]=df["y"].rolling(12,min_periods=12).mean()
    df["ema12"]=df["y"].ewm(span=12, adjust=False).mean()
    idx = df.index; hour = idx.hour + idx.minute/60.0
    df["sin_h"]=np.sin(2*np.pi*hour/24); df["cos_h"]=np.cos(2*np.pi*hour/24)
    df["weekday"]=idx.dayofweek; df["is_weekend"]=df["weekday"].isin([4,5,6]).astype(int)
    meal_feat = build_meal_features(idx, meal_times=meal_times, override_ts=override_ts)
    df = pd.concat([df, meal_feat], axis=1)
    return df.drop(columns=["y"]).dropna()

def predict_after_meal(readings, pkl_path, meal_ts, meal_key="lunch"):
    pkl = Path(pkl_path)
    if not pkl.exists(): return None
    with open(pkl, "rb") as f: model, meta = pickle.load(f)
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    if meal_ts not in s.index:
        s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="5min")).ffill()
    if meal_ts < s.index.min() + pd.Timedelta("2H"): return None
    X_all = make_features_only(
        s.loc[:meal_ts],
        meal_times=meta.get("meal_times", {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}),
        override_ts={meal_key: meal_ts}
    )
    if meal_ts not in X_all.index: meal_ts = X_all.index.max()
    x_row = X_all.loc[[meal_ts]]
    if "features" in meta:
        cols = [c for c in meta["features"] if c in x_row.columns]
        x_row = x_row.reindex(columns=cols, fill_value=0)
    yhat = float(model.predict(x_row)[0])
    return meta.get("horizon_min", 30), yhat

def plot_after_meal_curve(readings, meal_ts, meal_key="lunch",
                          pkls={30:"t1dm_lgbm_30m_meal.pkl",60:"t1dm_lgbm_60m_meal.pkl",120:"t1dm_lgbm_120m_meal.pkl"},
                          manual_points=None, history_hours=4, outpath="forecast_after_meal.png",
                          relative_axis=False):
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    hist = s.loc[meal_ts-pd.Timedelta(hours=history_hours): meal_ts]
    y0 = float(hist.iloc[-1]) if len(hist) else float(s.loc[:meal_ts].iloc[-1])
    points = {0: y0}
    for H,pkl in pkls.items():
        r = predict_after_meal(readings, pkl, meal_ts, meal_key=meal_key)
        if r is not None:
            h, yhat = r; points[int(h)] = float(yhat)
    if manual_points:
        for h,v in manual_points.items(): points[int(h)] = float(v)
    max_h = max([120]+list(points.keys()))
    grid_min = np.arange(0, max_h+1, 5)
    xp = np.array(sorted(points.keys())); yp = np.array([points[k] for k in sorted(points.keys())], float)
    curve = np.interp(grid_min, xp, yp)
    if relative_axis:
        hist_x = (hist.index - meal_ts).total_seconds()/60; t_grid = grid_min; xlab="식사 기준 상대시간(분)"
    else:
        hist_x = hist.index; t_grid = [meal_ts + pd.Timedelta(minutes=int(m)) for m in grid_min]; xlab="시간"
    plt.figure(figsize=(9,4))
    plt.plot(hist_x, hist.values, lw=1.5, label=f"식사 전 {history_hours}시간")
    plt.plot(t_grid, curve, "--", lw=2, label=f"식사 후 예측(0~{max_h}분)")
    for h,v in sorted(points.items()):
        if h>0:
            x = h if relative_axis else (meal_ts + pd.Timedelta(minutes=h))
            plt.plot([x],[v],"o",label=f"+{h}분")
    if relative_axis: plt.axvline(0, color="k", ls=":", lw=1)
    else:             plt.axvline(meal_ts, color="k", ls=":", lw=1)
    for th in [70,125,180,250]: plt.axhline(th, ls=":", lw=0.8)
    plt.title("식사 이후 120분 혈당 예측"); plt.xlabel(xlab); plt.ylabel("mg/dL"); plt.legend()
    plt.tight_layout(); plt.savefig(outpath, dpi=140); plt.show()
    return outpath

def predict_from_inputs(current_glucose, meal_ts, readings=None, meal_key="lunch",
                        history_hours=4,
                        pkls={30:"t1dm_lgbm_30m_meal.pkl",60:"t1dm_lgbm_60m_meal.pkl",120:"t1dm_lgbm_120m_meal.pkl"},
                        outpath="forecast_after_meal_from_inputs.png", relative_axis=False):
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    if readings is None:
        idx = pd.date_range(meal_ts-pd.Timedelta(hours=history_hours), meal_ts, freq="5min")
        s = pd.Series(float(current_glucose), index=idx, name="y")
    else:
        df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
        s = df.set_index("ds")["y"].asfreq("5min").ffill()
        if s.index[-1] < meal_ts:
            ext_idx = pd.date_range(s.index[-1]+pd.Timedelta("5min"), meal_ts, freq="5min")
            if len(ext_idx): s = pd.concat([s, pd.Series(float(s.iloc[-1]), index=ext_idx)], axis=0)
        elif s.index[-1] > meal_ts:
            s = s.loc[:meal_ts]
        s.iloc[-1] = float(current_glucose)
    readings2 = [{"ds": str(t), "y": float(v)} for t,v in s.items()]
    points = {0: float(current_glucose)}
    for H,pkl in pkls.items():
        r = predict_after_meal(readings2, pkl, meal_ts, meal_key=meal_key)
        if r is not None:
            h,yhat = r; points[int(h)] = float(yhat)
    if len(points)==1: points[120] = float(current_glucose)
    xs = np.array(sorted(points.keys())); ys = np.array([points[k] for k in sorted(points.keys())], float)
    yhat120 = float(points[120] if 120 in points else np.interp(120, xs, ys))
    if   yhat120 < 70:  risk = "위험(저혈당) 🔴"
    elif yhat120 < 125: risk = "주의 ⚠️"
    elif yhat120 < 180: risk = "경고 🚨"
    else:               risk = "위험(고혈당) 🔴"
    plot_path = plot_after_meal_curve(readings2, meal_ts, meal_key=meal_key,
                                      pkls=pkls, manual_points={k:v for k,v in points.items() if k>0},
                                      history_hours=history_hours, outpath=outpath, relative_axis=relative_axis)
    return {"predictions": {f"+{k}": round(points[k],1) for k in sorted(points) if k>0},
            "yhat_120": round(yhat120,1), "risk": risk, "plot_path": plot_path}

# ---------- (옵션) 570 파일에서 readings 만들기 ----------
readings = None
if Path("570-ws-training.xml").exists():
    dis_train = load_xml_glucose("570-ws-training.xml")
    recent = dis_train.tail(600)  # ≈ 50시간
    readings = [{"ds": str(r.ds), "y": float(r.y)} for _, r in recent.iterrows()]

# ---------- 예시 호출 ----------
res = predict_from_inputs(
    current_glucose=160,            # 현재 혈당 입력
    meal_ts="2022-01-15 13:10",     # 밥 먹은 시간 입력
    readings=readings,              # 없으면 None로 두세요(합성 히스토리)
    relative_axis=False             # True면 식사 기준 상대분(0~120) 축
)
print(res)


In [ ]:
# === 실제 유병자(570) 혈당: 식사시각 전후 실측 그래프 ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from pathlib import Path

# 폰트(있으면 자동 선택)
import matplotlib.font_manager as fm
cands = ["Malgun Gothic","NanumGothic","AppleGothic","Noto Sans CJK KR","DejaVu Sans"]
avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((c for c in cands if c in avail), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False

FILE = "588-ws-training.xml"     # 필요시 "570-ws-testing.xml"로 변경
MEAL_TS = "2021-08-31 12:48"     # 식사 시각(실제 확인하려는 시각)
H_BEFORE = "4H"                  # 식사 전 표시 구간
H_AFTER  = "2H"                  # 식사 후 표시 구간

def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try:    ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return (pd.DataFrame(rows)
              .dropna(subset=["ds"])
              .sort_values("ds")
              .reset_index(drop=True))

# 1) 데이터 로드
assert Path(FILE).exists(), f"{FILE} 없음"
df = load_xml_glucose(FILE)
s  = df.set_index("ds")["y"].asfreq("5min").ffill()

print(f"기간: {s.index.min()} ~ {s.index.max()}, 개수: {len(s):,}")
meal_ts = pd.to_datetime(MEAL_TS).floor("5min")

# 2) 식사 시각이 데이터 격자에 얼마나 근접한지 확인
if meal_ts not in s.index:
    # 가장 가까운 시각 찾기
    i = s.index.get_indexer([meal_ts], method="nearest")[0]
    nearest = s.index[i]
    delta_m = int(abs((nearest - meal_ts).total_seconds()) // 60)
    print(f"입력 식사시각 {meal_ts} → 가장 가까운 측정시각 {nearest} (차이 {delta_m}분)")
else:
    nearest = meal_ts
    delta_m = 0
    print(f"식사시각 {meal_ts}가 데이터 격자와 일치")

# 3) 전/후 구간 자르기
start = meal_ts - pd.Timedelta(H_BEFORE)
end   = meal_ts + pd.Timedelta(H_AFTER)
seg = s.loc[start:end].copy()

# 4) 간단 통계
y_meal   = float(s.loc[nearest])
y_post30 = float(s.loc[nearest + pd.Timedelta("30min")]) if nearest + pd.Timedelta("30min") in s.index else np.nan
y_post60 = float(s.loc[nearest + pd.Timedelta("60min")]) if nearest + pd.Timedelta("60min") in s.index else np.nan
y_post120= float(s.loc[nearest + pd.Timedelta("120min")]) if nearest + pd.Timedelta("120min") in s.index else np.nan
print({"meal_value": y_meal, "+30": y_post30, "+60": y_post60, "+120": y_post120})

# 5) 플롯
plt.figure(figsize=(9,4))
plt.plot(seg.index, seg.values, lw=1.8, label=f"실측({H_BEFORE}~{H_AFTER})")
plt.axvline(meal_ts, color="k", ls=":", lw=1.2, label="식사시각")
plt.scatter([nearest], [y_meal], zorder=3, s=50, color="tab:orange", label="가장 가까운 측정값")

# 임계선(선택)
for th in [70, 125, 180, 250]:
    plt.axhline(th, ls=":", lw=0.8)

plt.title("실제 유병자 혈당 (식사시각 전후)")
plt.xlabel("시간"); plt.ylabel("mg/dL"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# =========================================================
# 다른 유병자 XML로 "실측 vs 예측" 평가 (30/60/120분)
# - 저장된 LGBM 피클(t1dm_lgbm_30m/60m/120m_meal.pkl) 사용
# - 식사시각 리스트를 넣으면 각 시각마다 그래프+오차 출력
# =========================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle, xml.etree.ElementTree as ET
from pathlib import Path

# ---- 폰트(가능한 것 자동 선택) ----
import matplotlib.font_manager as fm
cands = ["Malgun Gothic","NanumGothic","AppleGothic","Noto Sans CJK KR","DejaVu Sans"]
avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = next((c for c in cands if c in avail), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False

# ====== <<<< 이 두 줄만 바꿔서 쓰면 됩니다 >>>> ======
NEW_FILE = "588-ws-training.xml"           # 다른 환자 XML 경로
MEALS = ["2021-08-31 12:48"]              # 평가할 식사시각 목록(여러 개 가능)
# ================================================

PKLS = {30:"t1dm_lgbm_30m_meal.pkl", 60:"t1dm_lgbm_60m_meal.pkl", 120:"t1dm_lgbm_120m_meal.pkl"}

# --- 공용: XML 로더 ---
def load_xml_glucose(path: str) -> pd.DataFrame:
    root = ET.parse(path).getroot()
    rows = []
    for ev in root.find("glucose_level").findall("event"):
        ts = ev.attrib["ts"]
        try:    ds = pd.to_datetime(ts, format="%d-%m-%Y %H:%M:%S", errors="raise")
        except: ds = pd.to_datetime(ts, errors="coerce")
        rows.append({"ds": ds, "y": float(ev.attrib["value"])})
    return (pd.DataFrame(rows).dropna(subset=["ds"]).sort_values("ds").reset_index(drop=True))

# --- 우리가 학습 때 쓰던 간단 피처셋(예측 시 1스텝만 필요) ---
def build_meal_features(index: pd.DatetimeIndex, meal_times=None, override_ts=None) -> pd.DataFrame:
    if meal_times is None:
        meal_times = {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}
    if override_ts is None: override_ts = {}
    base = pd.DataFrame({"t": index}).sort_values("t")
    feats, since_arrays = {}, []
    for key, hhmm in meal_times.items():
        days = pd.date_range(base["t"].min().floor("D")-pd.Timedelta(days=1),
                             base["t"].max().ceil("D"), freq="D")
        anchors = pd.Series(pd.to_datetime(days.strftime("%Y-%m-%d")+" "+hhmm), name=f"{key}_anc")
        if key in override_ts:
            ts = pd.to_datetime(override_ts[key])
            mask = anchors.dt.normalize() == ts.normalize()
            if mask.any(): anchors.loc[mask] = ts
        anc_df = anchors.to_frame()
        tmp = pd.merge_asof(base, anc_df, left_on="t", right_on=f"{key}_anc",
                            direction="backward", tolerance=pd.Timedelta("36H"))
        since = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60.0
        since = since.clip(lower=0).fillna(1440)
        feats[f"since_{key}_min"] = since.values
        feats[f"after_{key}_0_2h"] = ((since>=0)&(since<=120)).astype(int).values
        feats[f"after_{key}_2_4h"] = ((since>120)&(since<=240)).astype(int).values
        since_arrays.append(since.values)
    S = np.vstack(since_arrays); argmin = np.argmin(S, axis=0)
    for i,k in enumerate(meal_times.keys()):
        feats[f"last_is_{k}"] = (argmin==i).astype(int)
    feats["since_last_meal_min"] = S.min(axis=0)
    return pd.DataFrame(feats, index=index)

def make_features_only(s: pd.Series, meal_times=None, override_ts=None):
    df = pd.DataFrame({"y": s})
    for k in [1,2,3,6,12,24]: df[f"lag{k}"] = df["y"].shift(k)
    df["ma3"]=df["y"].rolling(3,min_periods=3).mean()
    df["ma12"]=df["y"].rolling(12,min_periods=12).mean()
    df["ema12"]=df["y"].ewm(span=12, adjust=False).mean()
    idx = df.index; hour = idx.hour + idx.minute/60.0
    df["sin_h"]=np.sin(2*np.pi*hour/24); df["cos_h"]=np.cos(2*np.pi*hour/24)
    df["weekday"]=idx.dayofweek; df["is_weekend"]=df["weekday"].isin([4,5,6]).astype(int)
    meal_feat = build_meal_features(idx, meal_times=meal_times, override_ts=override_ts)
    df = pd.concat([df, meal_feat], axis=1)
    return df.drop(columns=["y"]).dropna()

# --- 1회 예측: 특정 meal_ts에서 +H만큼 ---
def predict_after_meal(readings, pkl_path, meal_ts, meal_key="lunch"):
    pkl = Path(pkl_path)
    if not pkl.exists(): return None
    with open(pkl, "rb") as f:
        model, meta = pickle.load(f)
    df = pd.DataFrame(readings); df["ds"] = pd.to_datetime(df["ds"])
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()
    meal_ts = pd.to_datetime(meal_ts).floor("5min")
    if meal_ts not in s.index:
        s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="5min")).ffill()
    if meal_ts < s.index.min() + pd.Timedelta("2H"): return None
    X_all = make_features_only(s.loc[:meal_ts],
                meal_times=meta.get("meal_times", {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}),
                override_ts={meal_key: meal_ts})
    if meal_ts not in X_all.index: meal_ts = X_all.index.max()
    x = X_all.loc[[meal_ts]]
    if "features" in meta:
        cols = [c for c in meta["features"] if c in x.columns]
        x = x.reindex(columns=cols, fill_value=0)
    yhat = float(model.predict(x)[0])
    return meta.get("horizon_min", 30), yhat

def eval_and_plot(file_path, meal_ts, meal_key="lunch",
                  history_hours=4, pkls=PKLS, show=True):
    df = load_xml_glucose(file_path)
    s  = df.set_index("ds")["y"].asfreq("5min").ffill()

    smin, smax = s.index.min(), s.index.max()
    meal_ts = pd.to_datetime(meal_ts).floor("5min")

    # 1) 범위 확인 + 스냅(데이터 밖이면 경계로 이동)
    snapped_reason = None
    if meal_ts < smin:
        meal_ts = smin
        snapped_reason = "입력 시각이 데이터 시작보다 이릅니다 → 시작 시각으로 스냅"
    elif meal_ts > smax:
        meal_ts = smax
        snapped_reason = "입력 시각이 데이터 끝보다 늦습니다 → 끝 시각으로 스냅"
    else:
        # 5분 격자에서 가장 가까운 타임스탬프로 스냅
        i = s.index.get_indexer([meal_ts], method="nearest")[0]
        nearest = s.index[i]
        if nearest != meal_ts:
            snapped_reason = f"5분 격자에 맞춰 {nearest}로 스냅"
        meal_ts = nearest

    # 2) 히스토리(비면 시작점부터 확보)
    start = meal_ts - pd.Timedelta(hours=history_hours)
    if start < smin:
        start = smin
    hist = s.loc[start:meal_ts]
    if len(hist) == 0:
        # 최후 보정: 앞쪽 1시간이라도 보여주기
        hist = s.loc[smin:smin + pd.Timedelta("1H")]

    y0 = float(hist.iloc[-1])

    # 3) 예측 수집
    readings = [{"ds": str(t), "y": float(v)} for t, v in s.loc[:meal_ts].items()]
    preds, trues, abs_errs = {}, {}, {}
    for H, p in pkls.items():
        r = predict_after_meal(readings, p, meal_ts, meal_key=meal_key)
        if r is None:
            continue
        _, yhat = r
        preds[H] = yhat
        t = meal_ts + pd.Timedelta(minutes=H)
        trues[H] = float(s[t]) if t in s.index else np.nan
        abs_errs[H] = abs(trues[H] - preds[H]) if not np.isnan(trues[H]) else np.nan

    # 4) 예측 곡선(보간)
    pts = {0: y0, **preds}
    xs = np.array(sorted(pts.keys())); ys = np.array([pts[k] for k in xs], float)
    grid = np.arange(0, max(120, xs.max() if len(xs) else 120) + 1, 5)
    curve = np.interp(grid, xs, ys) if len(xs) else np.full_like(grid, y0, dtype=float)

    # 5) 플롯
    if show:
        plt.figure(figsize=(9,4))
        plt.plot(hist.index, hist.values, lw=1.6, label=f"식사 전 {history_hours}시간")
        t_grid = [meal_ts + pd.Timedelta(minutes=int(m)) for m in grid]
        plt.plot(t_grid, curve, "--", lw=2, label="식사 후 예측(0~120분)")
        for H, v in preds.items():
            plt.plot([meal_ts + pd.Timedelta(minutes=H)], [v], "o", label=f"예측 +{H}분")
        for H, tv in trues.items():
            if not np.isnan(tv):
                plt.plot([meal_ts + pd.Timedelta(minutes=H)], [tv], "s", label=f"실측 +{H}분")
        plt.axvline(meal_ts, color="k", ls=":", lw=1)
        for th in [70, 125, 180, 250]:
            plt.axhline(th, ls=":", lw=0.8)
        title = f"{file_path} | 식사시각 {meal_ts}"
        if snapped_reason: title += f"  ({snapped_reason})"
        plt.title(title)
        plt.xlabel("시간"); plt.ylabel("mg/dL"); plt.legend()
        plt.tight_layout(); plt.show()

    mae = round(np.nanmean(list(abs_errs.values())), 2) if len(abs_errs) else np.nan
    return {
        "meal_ts": str(meal_ts),
        "range": (str(smin), str(smax)),
        "preds": preds,
        "trues": trues,
        "abs_errs": {k: round(v, 1) for k, v in abs_errs.items() if not np.isnan(v)},
        "MAE": mae
    }


In [ ]:
results = []
for ts in MEALS:
    out = eval_and_plot(NEW_FILE, ts, meal_key="lunch", history_hours=4, pkls=PKLS, show=True)
    print(out)
    results.append(out)

In [ ]:
def predict_from_inputs(current_glucose,
                        meal_ts,
                        readings=None,
                        meal_key="lunch",
                        history_hours=4,
                        pkls=None,
                        outpath="forecast_after_meal.png",
                        relative_axis=False):
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle
    from pathlib import Path

    if pkls is None:
        pkls = {30: "t1dm_lgbm_30m_meal.pkl",
                60: "t1dm_lgbm_60m_meal.pkl",
                120:"t1dm_lgbm_120m_meal.pkl"}

    meal_ts = pd.to_datetime(meal_ts).floor("5min")

    # 1) readings → 5분 고정 시계열 s
    if readings is not None:
        df = pd.DataFrame(readings).copy()
        df["ds"] = pd.to_datetime(df["ds"])
        s = df.set_index("ds")["y"].asfreq("5min").ffill()

        # meal_ts가 범위 밖이면 안전하게 처리
        if meal_ts < s.index.min() or meal_ts > s.index.max():
            # 합성 히스토리(현재 혈당으로 평평하게)
            s = pd.Series(current_glucose,
                          index=pd.date_range(meal_ts - pd.Timedelta(hours=history_hours),
                                              meal_ts, freq="5min"),
                          dtype=float)
        else:
            # meal_ts까지 잘라 쓰기 + 빈 경우 대비
            s = s.loc[:meal_ts]
            if len(s) == 0:
                s = pd.Series(current_glucose,
                              index=pd.date_range(meal_ts - pd.Timedelta(hours=history_hours),
                                                  meal_ts, freq="5min"),
                              dtype=float)
    else:
        # readings가 아예 없으면 합성 히스토리
        s = pd.Series(current_glucose,
                      index=pd.date_range(meal_ts - pd.Timedelta(hours=history_hours),
                                          meal_ts, freq="5min"),
                      dtype=float)

    # meal_ts 타임스탬프를 반드시 포함시키고 현재 혈당 반영
    if meal_ts not in s.index:
        s = s.reindex(s.index.union([meal_ts])).sort_index().ffill()
    s.loc[meal_ts] = float(current_glucose)

    # 2) 예측 포인트 수집
    def _predict_one(pkl_path, series, horizon_min):
        pkl = Path(pkl_path)
        if not pkl.exists(): return None
        with open(pkl, "rb") as f:
            model, meta = pickle.load(f)
        # 학습 때 쓰인 피처 생성기와 동일한 make_features_only 사용했다고 가정
        def build_meal_features(index, meal_times=None, override_ts=None):
            if meal_times is None:
                meal_times = {"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}
            if override_ts is None: override_ts = {}
            base = pd.DataFrame({"t": index}).sort_values("t")
            feats, arrs = {}, []
            for key, hhmm in meal_times.items():
                days = pd.date_range(base["t"].min().floor("D")-pd.Timedelta(days=1),
                                     base["t"].max().ceil("D"), freq="D")
                anchors = pd.Series(pd.to_datetime(days.strftime("%Y-%m-%d")+" "+hhmm), name=f"{key}_anc")
                if key in override_ts:
                    ts = pd.to_datetime(override_ts[key])
                    m = anchors.dt.normalize() == ts.normalize()
                    if m.any(): anchors.loc[m] = ts
                tmp = pd.merge_asof(base, anchors.to_frame(), left_on="t", right_on=f"{key}_anc",
                                    direction="backward", tolerance=pd.Timedelta("36H"))
                since = (base["t"] - tmp[f"{key}_anc"]).dt.total_seconds()/60
                since = since.clip(lower=0).fillna(1440)
                feats[f"since_{key}_min"] = since.values
                feats[f"after_{key}_0_2h"] = ((since>=0)&(since<=120)).astype(int).values
                feats[f"after_{key}_2_4h"] = ((since>120)&(since<=240)).astype(int).values
                arrs.append(since.values)
            S = np.vstack(arrs); argmin = np.argmin(S, axis=0)
            keys = list({"breakfast":"08:00","lunch":"13:00","dinner":"19:00"}.keys())
            for i,k in enumerate(keys):
                feats[f"last_is_{k}"] = (argmin==i).astype(int)
            feats["since_last_meal_min"] = S.min(axis=0)
            return pd.DataFrame(feats, index=index)

        def make_features_only(series, meal_times=None, override_ts=None):
            df2 = pd.DataFrame({"y": series})
            for k in [1,2,3,6,12,24]:
                df2[f"lag{k}"] = df2["y"].shift(k)
            df2["ma3"] = df2["y"].rolling(3, min_periods=3).mean()
            df2["ma12"]= df2["y"].rolling(12, min_periods=12).mean()
            df2["ema12"]=df2["y"].ewm(span=12, adjust=False).mean()
            idx = df2.index; hour = idx.hour + idx.minute/60.0
            df2["sin_h"]=np.sin(2*np.pi*hour/24); df2["cos_h"]=np.cos(2*np.pi*hour/24)
            df2["weekday"]=idx.dayofweek; df2["is_weekend"]=df2["weekday"].isin([4,5,6]).astype(int)
            meal_feat = build_meal_features(idx, meal_times=meta.get("meal_times"), override_ts={meal_key: meal_ts})
            df2 = pd.concat([df2, meal_feat], axis=1).drop(columns=["y"]).dropna()
            return df2

        X_all = make_features_only(series)
        if meal_ts not in X_all.index:
            return None
        x = X_all.loc[[meal_ts]]
        if "features" in meta:
            x = x.reindex(columns=[c for c in meta["features"] if c in x.columns], fill_value=0)
        return float(model.predict(x)[0])

    points = {0: float(current_glucose)}
    preds = {}
    for H, pkl in pkls.items():
        yhat = _predict_one(pkl, s, H)
        if yhat is not None:
            preds[H] = yhat
            points[H] = yhat

    # 3) 곡선 만들고 그리기
    import matplotlib.pyplot as plt
    xs = sorted(points.keys()); ys = [points[h] for h in xs]
    grid = np.arange(0, max(120, max(xs)) + 1, 5)
    curve = np.interp(grid, xs, ys)
    if relative_axis:
        t_hist = s.index
        t_grid = grid  # 0~120 분
        x_hist = (t_hist - meal_ts).total_seconds()/60
    else:
        t_hist = s.index
        t_grid = [meal_ts + pd.Timedelta(minutes=int(m)) for m in grid]
        x_hist = t_hist

    plt.figure(figsize=(9,4))
    plt.plot(x_hist, s.values, lw=1.6, label=f"식사 전 {history_hours}시간")
    plt.plot(t_grid, curve, "--", lw=2, label="식사 후 예측(0~120분)")
    for H in [30,60,120]:
        if H in preds:
            x = H if relative_axis else (meal_ts + pd.Timedelta(minutes=H))
            plt.plot([x], [preds[H]], "o", label=f"+{H}분")
    if relative_axis:
        plt.axvline(0, color="k", ls=":", lw=1); plt.xlabel("식사 기준 경과 시간(분)")
    else:
        plt.axvline(meal_ts, color="k", ls=":", lw=1); plt.xlabel("시간")
    for th in [70,125,180,250]:
        plt.axhline(th, ls=":", lw=0.8)
    plt.ylabel("mg/dL"); plt.legend(); plt.tight_layout()
    plt.savefig(outpath, dpi=140); plt.show()

    return {"meal_ts": str(meal_ts), "preds": preds, "figure": outpath}


In [ ]:
res = predict_from_inputs(
    current_glucose=160,
    meal_ts="2021-08-31 12:48",
    readings=readings,      # 있으면 그대로
    relative_axis=False
)
print(res)
